# Model Evaluation for Deployment

This notebook evaluates a candidate model version before deployment.

**Purpose:**
- Load model from Unity Catalog
- Evaluate on validation/test dataset
- Calculate MAP@12 and other metrics
- Log evaluation results to MLflow

**Note:** This notebook should only be run in a Databricks Job, as part of MLflow 3.0 Deployment Jobs.

## Setup

In [0]:
import sys

# Add project root to path (go up 2 levels from deployment_step/)
sys.path.append("../../")

import pandas as pd
import mlflow

from config.catalog_config import get_table_config
from utils.data_utils import load_delta_table
from utils.evaluation_utils import calculate_map_at_k, log_evaluation_metrics

In [0]:
# Define widgets for job parameters
dbutils.widgets.text("model_name", "")
dbutils.widgets.text("model_version", "")
dbutils.widgets.text("catalog_name", "shared")
dbutils.widgets.text("schema_name", "fashion_recommendations")
dbutils.widgets.text("evaluation_dataset", "val")  # 'val' or 'test'

In [0]:
# Get parameters
model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
evaluation_dataset = dbutils.widgets.get("evaluation_dataset")

print(f"Evaluating Model: {model_name}")
print(f"Version: {model_version}")
print(f"Catalog: {catalog_name}")
print(f"Schema: {schema_name}")
print(f"Evaluation Dataset: {evaluation_dataset}")

## Load Evaluation Data

In [0]:
# Get table names
tables = get_table_config(catalog_name, schema_name)

# Load ground truth based on evaluation dataset parameter
if evaluation_dataset == "test":
    ground_truth_table = tables.TEST_GROUND_TRUTH_SILVER
    print(f"Using TEST dataset: {ground_truth_table}")
else:
    ground_truth_table = tables.VAL_GROUND_TRUTH_SILVER
    print(f"Using VALIDATION dataset: {ground_truth_table}")

# Load ground truth as pandas DataFrame
ground_truth_df = load_delta_table(ground_truth_table).toPandas()
print(f"Loaded ground truth: {len(ground_truth_df):,} customers")

# Load customer features (includes age_group and other derived features)
print(f"\nLoading customer features from: {tables.CUSTOMER_FEATURES}")
try:
    customer_features_df = spark.table(tables.CUSTOMER_FEATURES).toPandas()
    print(f"Loaded customer features: {len(customer_features_df):,} customers")
except Exception as e:
    print(f"Customer features table not found, will use basic customer data")
    print(f"Loading customers from: {tables.CUSTOMERS_SOURCE}")
    customer_features_df = spark.table(tables.CUSTOMERS_SOURCE).toPandas()
    # Create age_group feature if not present
    if "age_group" not in customer_features_df.columns:
        customer_features_df["age_group"] = pd.cut(
            customer_features_df["age"],
            bins=[0, 25, 35, 45, 55, 65, float('inf')],
            labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
            right=False
        )
    print(f"Loaded customer data: {len(customer_features_df):,} customers")

## Load Model and Generate Predictions

In [0]:
# Load model from Unity Catalog
model_uri = f"models:/{model_name}/{model_version}"
print(f"Loading model: {model_uri}")

# Load model using MLflow
model = mlflow.pyfunc.load_model(model_uri)
print("Model loaded successfully")

In [0]:
# Generate predictions for evaluation customers
print("Generating predictions...")

# Get unique customer IDs from ground truth
eval_customer_ids = ground_truth_df[["customer_id"]].drop_duplicates()

# Join with customer features to get all required features (like age_group)
# This matches the approach in batch_inference.ipynb
eval_customers_with_features = eval_customer_ids.merge(
    customer_features_df,
    on="customer_id",
    how="left"
)

print(f"Evaluation customers with features: {len(eval_customers_with_features):,}")

# Predict using model (model will use only the columns it needs based on its signature)
predictions_pd = model.predict(eval_customers_with_features)

# Ensure predicted articles are strings
predictions_pd["predicted_articles"] = predictions_pd["predicted_articles"].apply(lambda x: [str(i) for i in x])

# Keep as pandas DataFrame
predictions_df = predictions_pd[["customer_id", "predicted_articles"]]
print(f"Generated predictions: {len(predictions_df):,} customers")

## Evaluate Model Performance

In [0]:
# Start MLflow run for evaluation
with mlflow.start_run(run_name="deployment_evaluation") as run:
    # Log model info
    mlflow.set_tag("model_name", model_name)
    mlflow.set_tag("model_version", model_version)
    mlflow.set_tag("evaluation_dataset", evaluation_dataset)
    mlflow.set_tag("stage", "deployment_evaluation")
    
    # Calculate and log metrics
    print("\n" + "="*60)
    print(f"EVALUATING {model_name} v{model_version}")
    print("="*60)
    
    metrics = log_evaluation_metrics(
        predictions_df,
        ground_truth_df,
        model_name=f"{model_name} v{model_version}",
        k=12
    )
    
    # Save run ID
    evaluation_run_id = run.info.run_id
    print(f"\nEvaluation MLflow run ID: {evaluation_run_id}")
    print(f"MAP@12: {metrics['map@12']:.6f}")

## Summary

In [0]:
print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)
print(f"Model: {model_name}")
print(f"Version: {model_version}")
print(f"MAP@12: {metrics['map@12']:.6f}")
print(f"Evaluated on: {evaluation_dataset} dataset")
print(f"Customers evaluated: {metrics['num_customers']:,}")
print("="*60)
print("\n✓ Model evaluation successful. Ready for approval step.")